# Validação do DemDS

Comparação do algoritmo original em C++ com reimplementação em Python baseado no resultado por iterações

## 1. Configuração

O lote usa os edgelists de `RAW_DATA_DIR` em `data/raw/`. Para testar um arquivo, escolha `instance_name` e selecione o modo de instância única na seção 3. Ajuste `CPP_DIR` se `DemDS.h` e o binário original estiverem em outro local. O teste por iterações usa `g++` disponível no ambiente do kernel.

In [15]:
import gc
import subprocess
import sys
from operator import index
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from tempfile import TemporaryDirectory
from time import perf_counter

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
for directory in (cwd, *cwd.parents):
    src = directory / "src"
    if (src / "complex_networks").is_dir():
        for import_path in (directory, src):
            if str(import_path) not in sys.path:
                sys.path.insert(0, str(import_path))
        break

from src.complex_networks.algorithms.mds.demds import mds_solver_demds
from src.complex_networks.config import PROJECT_ROOT, RAW_DATA_DIR
from src.complex_networks.io import load_edgelist, preprocess
from src.complex_networks.utils import iter_files

instance_name = "amazon0302"
GRAPH_PATH = Path(RAW_DATA_DIR) / f"{instance_name}.txt.gz"
CPP_DIR = Path(PROJECT_ROOT) / "libs" / "DemDS"
CPP_HEADER = CPP_DIR / "DemDS.h"
CPP_BIN = CPP_DIR / "DemDS"
SEED = 1
CPP_ALPHA_ARG = 90

## 2. Funções auxiliares

Os tempos de parede são medidos com `perf_counter()`. A leitura do edgelist, a conversão da entrada para C++ e a validação da solução ficam fora da medição.

In [16]:
def is_dominating_set(G, solution):
    return all(v in G for v in solution) and all(
        v in solution or any(u in solution for u in G[v])
        for v in G
    )


def run_python(G, *, seed, **budget):
    started = perf_counter()
    solution = mds_solver_demds(G, seed=seed, **budget)
    return solution, perf_counter() - started


def write_cpp_instance(G, path):
    """Escreve a entrada C++ e devolve os rótulos na ordem dos IDs internos."""
    if G.is_directed() or G.is_multigraph():
        raise ValueError("Use um grafo simples e não direcionado.")

    labels = tuple(G)
    ids = {v: i for i, v in enumerate(labels, start=1)}
    with path.open("w", encoding="ascii") as stream:
        stream.write(f"{len(labels)} {G.number_of_edges()}\n")
        for u, v in G.edges():
            if u == v:
                raise ValueError("Remova os laços do grafo com preprocess.")
            stream.write(f"{ids[u]} {ids[v]}\n")
    return labels


def run_cpp(executable, instance, *, seed, time_limit, alpha_arg):
    args = [str(executable), instance.name, str(seed), str(time_limit), str(alpha_arg)]
    started = perf_counter()
    result = subprocess.run(
        args, cwd=instance.parent, capture_output=True, encoding="utf-8", errors="replace"
    )
    elapsed = perf_counter() - started
    row = {"output_cpp": None, "time_cpp_s": round(elapsed, 3), "error_cpp": None}

    if result.returncode != 0:
        row["error_cpp"] = f"returncode={result.returncode} {result.stderr.strip()}".strip()
        return row

    try:
        _, size, _ = result.stdout.strip().splitlines()[-1].rsplit(",", 2)
        row["output_cpp"] = int(size)
    except (IndexError, ValueError):
        row["error_cpp"] = f"Saída inesperada do C++: {result.stdout.strip()!r}"
    return row

## 3. Comparar os conjuntos com o mesmo orçamento de iterações

Cada par em `TESTES_FIXOS` contém a semente e o limite de iterações para **ambas** as implementações. A célula de execução permite validar todos os `.txt.gz` de `RAW_DATA_DIR` ou apenas o `instance_name` selecionado.

O executável auxiliar chama as rotinas de `DemDS.h` e registra o contador e os vértices do melhor conjunto. No auxiliar, o relógio fica fixo para desativar o corte por tempo e as podas periódicas, como no modo `max_iterations` do Python. A poda inicial é executada nos dois.

`mesmo_conjunto` compara diretamente os rótulos dos vértices. Confira também `valido_cpp` e `valido_python`. Uma falha do C++ aparece em `error_cpp`, com a igualdade indisponível para aquela execução.

In [17]:
CPP_VALIDATOR = r"""
#include <ctime>
#include <fstream>
#include <string>

static std::clock_t validation_clock() { return 0; }
#define clock validation_clock
#include "DemDS.h"
#undef clock

int main(int argc, char* argv[]) {
    if (argc != 5) return 2;
    std::ifstream input(argv[1]);
    if (!input) return 1;

    seed = std::stoi(argv[2]);
    nbIter = std::stoll(argv[3]);
    alpha = std::stoi(argv[4]);
    gen.seed(seed);

    MDSP solver;
    input >> solver;
    solver.ini_greed();
    solver.calc_L2();
    solver.calc_L3();
    solver.calc_score();
    solver.improve(0);
    solver.begin_time = validation_clock();

    auto best = solver.X2;
    const auto fixed_count = static_cast<std::size_t>(solver.ini_num);
    long long iter = 0;
    long long best_iter = 0;
    while (iter < nbIter && solver.X2.size() > fixed_count) {
        solver.remove_vertex(solver.X2[fixed_count], iter);
        current_X = solver.X2.size();
        if (solver.x_minus != 0 && solver.X2.size() == fixed_count) break;
        if (!solver.Dem_kDS(iter)) break;

        best = solver.X2;
        best_iter = iter;
    }

    std::cout << iter << ' ' << best_iter << '\n';
    for (int v : best) std::cout << v << ' ';
    std::cout << '\n';
}
"""


def compile_cpp_validator(header, directory):
    header = header.resolve()
    if not header.is_file():
        raise FileNotFoundError(f"Ajuste CPP_HEADER. Arquivo ausente em {header}")

    source = directory / "validator.cpp"
    source.write_text(CPP_VALIDATOR, encoding="utf-8")
    executable = directory / "DemDS_validacao"
    try:
        result = subprocess.run(
            [
                "g++", "-std=c++17", "-O3", "-D_GLIBCXX_ASSERTIONS",
                "-I", str(header.parent), str(source), "-o", str(executable),
            ],
            capture_output=True, encoding="utf-8", errors="replace",
        )
    except FileNotFoundError as error:
        raise RuntimeError("O teste por iterações precisa do g++ no ambiente do kernel.") from error
    if result.returncode:
        raise RuntimeError(f"Falha ao compilar a validação C++.\n{result.stderr}")
    return executable


def run_cpp_fixed(executable, instance, labels, *, seed, iterations, alpha_arg):
    started = perf_counter()
    result = subprocess.run(
        [str(executable), instance.name, str(seed), str(iterations), str(alpha_arg)],
        cwd=instance.parent, capture_output=True, encoding="utf-8", errors="replace",
    )
    row = {
        "iteracoes_cpp": None,
        "melhor_iteracao_cpp": None,
        "output_cpp": None,
        "time_cpp_s": round(perf_counter() - started, 3),
        "error_cpp": None,
    }
    if result.returncode:
        row["error_cpp"] = f"returncode={result.returncode} {result.stderr.strip()}".strip()
        return None, row

    try:
        counters, vertex_line = result.stdout.splitlines()
        iter_done, best_iter = map(int, counters.split())
        ids = list(map(int, vertex_line.split()))
        if not 0 <= best_iter <= iter_done <= iterations:
            raise ValueError("Contador de iterações fora do intervalo esperado.")
        if any(v < 0 or v >= len(labels) for v in ids):
            raise ValueError("ID de vértice fora do intervalo do grafo.")
        # O leitor C++ converte os IDs exportados de 1 para 0.
        solution = {labels[v] for v in ids}
        if len(solution) != len(ids):
            raise ValueError("A solução C++ contém IDs repetidos.")
    except ValueError as error:
        row["error_cpp"] = f"Saída C++ inválida. {error}"
        return None, row

    row.update(iteracoes_cpp=iter_done, melhor_iteracao_cpp=best_iter, output_cpp=len(solution))
    return solution, row

In [18]:
def _iter_fixed_results(graph_path, executable, instance, cases, *, alpha_arg):
    G = None
    try:
        G = load_edgelist(graph_path)
        G = preprocess(G)
        labels = write_cpp_instance(G, instance)
        for seed, iterations in cases:
            print(f"{graph_path.name} | semente {seed} | limite {iterations} iterações", flush=True)
            solution_cpp, row_cpp = run_cpp_fixed(
                executable, instance, labels, seed=seed, iterations=iterations, alpha_arg=alpha_arg,
            )
            solution_py, elapsed = run_python(G, seed=seed, max_iterations=iterations)
            yield {
                "grafo": graph_path.name.removesuffix(".txt.gz"),
                "seed": seed,
                "max_iterations": iterations,
                **row_cpp,
                "output_python": len(solution_py),
                "time_python_s": round(elapsed, 3),
                "valido_cpp": is_dominating_set(G, solution_cpp) if solution_cpp is not None else None,
                "valido_python": is_dominating_set(G, solution_py),
                "mesmo_conjunto": solution_cpp == solution_py if solution_cpp is not None else None,
            }
            del solution_cpp, solution_py
    finally:
        del G
        gc.collect()


def check_fixed_budgets_many(graph_paths, header, cases, *, alpha_arg, continue_on_error=True):
    cases = [(index(seed), index(iterations)) for seed, iterations in cases]
    if any(not -(1 << 31) <= seed < (1 << 31) for seed, _ in cases):
        raise ValueError("A semente precisa caber em um inteiro C++ de 32 bits.")
    if any(not 0 <= iterations < (1 << 63) for _, iterations in cases):
        raise ValueError("Use um limite de iterações não negativo de até 2**63 - 1.")

    paths = tuple(graph_paths)
    rows = []
    if not paths or not cases:
        return pd.DataFrame(rows)

    with TemporaryDirectory(prefix="demds-validacao-") as temporary_dir:
        directory = Path(temporary_dir)
        executable = compile_cpp_validator(header, directory)
        instance = directory / "instance.txt"
        for graph_path in paths:
            try:
                rows.extend(_iter_fixed_results(
                    graph_path, executable, instance, cases, alpha_arg=alpha_arg
                ))
            except Exception as error:
                if not continue_on_error:
                    raise
                message = f"{type(error).__name__}: {error}"
                print(f"{graph_path.name} | {message}", flush=True)
                rows.append({
                    "grafo": graph_path.name.removesuffix(".txt.gz"),
                    "mesmo_conjunto": None,
                    "error_graph": message,
                })
    return pd.DataFrame(rows)


def check_fixed_budgets(graph_path, header, cases, *, alpha_arg):
    return check_fixed_budgets_many(
        (graph_path,), header, cases, alpha_arg=alpha_arg, continue_on_error=False
    )

### Executar a validação

Com `RODAR_LOTE = True`, o notebook percorre os `.txt.gz` de `RAW_DATA_DIR` com `iter_files`. `"size"` ordena pelo tamanho compactado e `"name"` ordena pelo nome. Com `False`, a célula testa apenas `instance_name`.

O lote compila o auxiliar uma vez, libera cada grafo após a validação e reúne os resultados em `resultados_iteracoes`. Uma falha ao processar um grafo aparece em `error_graph` e o lote segue para o próximo arquivo.

In [19]:
TESTES_FIXOS = [(1, 10_000_000)]
RODAR_LOTE = True
ORDEM_ARQUIVOS = "size"

if TESTES_FIXOS:
    if RODAR_LOTE:
        graph_paths = iter_files(RAW_DATA_DIR, "*.txt.gz", order=ORDEM_ARQUIVOS)
        resultados_iteracoes = check_fixed_budgets_many(
            graph_paths, CPP_HEADER, TESTES_FIXOS, alpha_arg=CPP_ALPHA_ARG
        )
    else:
        resultados_iteracoes = check_fixed_budgets(
            GRAPH_PATH, CPP_HEADER, TESTES_FIXOS, alpha_arg=CPP_ALPHA_ARG
        )
    display(resultados_iteracoes)

gemsec-Facebook-tvshow.txt.gz | semente 1 | limite 10000000 iterações
p2p-Gnutella04.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Facebook-politician.txt.gz | semente 1 | limite 10000000 iterações
p2p-Gnutella25.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Facebook-company.txt.gz | semente 1 | limite 10000000 iterações
p2p-Gnutella24.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Facebook-public_figure.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Facebook-government.txt.gz | semente 1 | limite 10000000 iterações
p2p-Gnutella30.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Facebook-athletes.txt.gz | semente 1 | limite 10000000 iterações
p2p-Gnutella31.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Deezer-ro.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Facebook-new_sites.txt.gz | semente 1 | limite 10000000 iterações
gemsec-Deezer-hu.txt.gz | semente 1 | limite 10000000 iterações
email-EuAll.txt.gz | semente 1 | limite 10000000 it

,grafo,seed,max_iterations,iteracoes_cpp,melhor_iteracao_cpp,output_cpp,time_cpp_s,error_cpp,output_python,time_python_s,valido_cpp,valido_python,mesmo_conjunto
0,gemsec-Facebook-tvshow,1,10000000,10000000.0,306.0,719.0,7.231,NaN,719,81.421,True,True,True
1,p2p-Gnutella04,1,10000000,10000000.0,15275.0,2227.0,6.858,NaN,2227,78.369,True,True,True
2,gemsec-Facebook-politician,1,10000000,10000000.0,8178.0,736.0,10.070,NaN,736,94.344,True,True,True
3,p2p-Gnutella25,1,10000000,10000000.0,520.0,4519.0,5.258,NaN,4519,62.736,True,True,True
4,gemsec-Facebook-company,1,10000000,10000000.0,46143.0,2717.0,7.649,NaN,2717,78.790,True,True,True
5,p2p-Gnutella24,1,10000000,10000000.0,1179.0,5418.0,5.207,NaN,5418,67.912,True,True,True
6,gemsec-Facebook-public_figure,1,10000000,10000000.0,11797.0,2086.0,9.839,NaN,2086,94.141,True,True,True
7,gemsec-Facebook-government,1,10000000,10000000.0,701916.0,686.0,15.122,NaN,686,128.682,True,True,True
8,p2p-Gnutella30,1,10000000,10000000.0,87.0,7169.0,5.450,NaN,7169,88.281,True,True,True
9,gemsec-Facebook-athletes,1,10000000,10000000.0,23725.0,1636.0,13.344,NaN,1636,111.031,True,True,True


## 4. Repetir a comparação por tempo

Defina `LIMITES_TEMPO = [180]` para repetir o teste de 180 s ou `[30, 180]` para comparar os dois limites. Cada limite roda C++ e Python sequencialmente. A tabela mostra as cardinalidades, a validade da solução Python e o tempo de parede de cada execução.

In [ ]:
def compare_time_limits(graph_path, executable, limits, *, seed, alpha_arg):
    executable = executable.resolve()
    if not executable.is_file():
        raise FileNotFoundError(f"Ajuste CPP_BIN. Executável ausente em {executable}")

    rows = []
    G = None
    try:
        G = load_edgelist(graph_path)
        G = preprocess(G)
        with TemporaryDirectory(prefix="demds-") as directory:
            instance = Path(directory) / "instance.txt"
            write_cpp_instance(G, instance)
            for seconds in limits:
                print(f"C++ com limite de {seconds} s", flush=True)
                row_cpp = run_cpp(
                    executable, instance, seed=seed, time_limit=seconds, alpha_arg=alpha_arg
                )
                print(f"Python com limite de {seconds} s", flush=True)
                solution, elapsed = run_python(G, seed=seed, time_limit=seconds)
                rows.append({
                    "grafo": graph_path.name.removesuffix(".txt.gz"),
                    "seed": seed,
                    "time_limit_s": seconds,
                    "cpp_alpha_arg": alpha_arg,
                    **row_cpp,
                    "output_python": len(solution),
                    "time_python_s": round(elapsed, 3),
                    "valido_python": is_dominating_set(G, solution),
                })
                del solution
        return pd.DataFrame(rows)
    finally:
        del G
        gc.collect()


LIMITES_TEMPO = []  # Use [180] ou [30, 180].
if LIMITES_TEMPO:
    display(compare_time_limits(
        GRAPH_PATH, CPP_BIN, LIMITES_TEMPO, seed=SEED, alpha_arg=CPP_ALPHA_ARG
    ))